In [1]:
import pretty_midi
import os
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import datetime
from torch.utils.data import random_split
import torch.nn.functional as F
from tqdm import tqdm # Great for progress bars during processing
import glob

## ARCHITECTURE

In [2]:
class MusicSotaTransformer(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_layers=6, max_seq_len=128):
        super().__init__()
        
        # 1. Vocabulary
        self.pitch_embed = nn.Embedding(128, d_model)   # 128 MIDI Notes
        self.step_embed = nn.Embedding(100, d_model)    # 100 Time Bins
        self.dur_embed = nn.Embedding(100, d_model)     # 100 Duration-Bins
        self.vel_embed = nn.Embedding(32, d_model)      # 32 Velocity-Bins
        
        # 2. Merge into one Layer
        self.feature_merger = nn.Linear(d_model * 4, d_model)
        
        # 3. Position Encoding: Significance for note at the start and at teh end
        self.pos_embed = nn.Parameter(torch.zeros(1, max_seq_len, d_model))
        
        # 4. Transformer Core (GPT-style Decoder)
        #
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=1024, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(decoder_layer, num_layers=num_layers)
        
        # 5. seperate outputs for each value (Multi-Task Learning)
        #
        self.head_pitch = nn.Linear(d_model, 128)
        self.head_step = nn.Linear(d_model, 100)
        self.head_dur = nn.Linear(d_model, 100)
        self.head_vel = nn.Linear(d_model, 32)

    def forward(self, p, s, d, v):
        # get embeddings
        p_e = self.pitch_embed(p)
        s_e = self.step_embed(s)
        d_e = self.dur_embed(d)
        v_e = self.vel_embed(v)
        
        # combine features
        x = torch.cat((p_e, s_e, d_e, v_e), dim=-1)
        
        # 1. Get current sequence length (e.g., 64)
        seq_len = x.size(1)
        
        # 2. FIX: Slice the 128-length pos_embed to match the 64-length x
        x = self.feature_merger(x) + self.pos_embed[:, :seq_len, :]
        
        # Transformer
        x = self.transformer(x)
        
        # use last token for prediction
        last_token = x[:, -1, :]
        
        # prediction for every output head
        return (
            self.head_pitch(last_token),
            self.head_step(last_token),
            self.head_dur(last_token),
            self.head_vel(last_token)
        )

### Alternative

In [3]:
class SotaMusicTransformerCNN(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_layers=6, window_size=64):
        super().__init__()
        self.d_model = d_model
        
        # 1. Word Embeddings
        self.pitch_emb = nn.Embedding(128, d_model)
        self.step_emb = nn.Embedding(100, d_model)
        self.dur_emb = nn.Embedding(100, d_model)
        self.vel_emb = nn.Embedding(32, d_model)
        
        # 2. Feature Fusion & Bottleneck
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 4, d_model * 2),
            nn.GELU(),
            nn.Linear(d_model * 2, d_model)
        )
        
        # 3. Convolutional Front-End (The "Motif Scanner")
        # We use Conv1d to find local melodic and rhythmic patterns
        self.conv_frontend = nn.Sequential(
            nn.Conv1d(d_model, d_model, kernel_size=3, padding=1),
            nn.GroupNorm(8, d_model), # Stable normalization for small batches
            nn.GELU(),
            nn.Conv1d(d_model, d_model, kernel_size=3, padding=1),
            nn.GroupNorm(8, d_model),
            nn.GELU()
        )
        
        # 4. Learned Positional Encoding
        self.pos_emb = nn.Parameter(torch.randn(1, window_size, d_model))
        
        # 5. Transformer Decoder Stack (Pre-Norm style)
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=d_model * 4, 
            dropout=0.1, 
            activation=nn.GELU(),
            batch_first=True,
            norm_first=True # Pre-Norm for better stability
        )
        self.transformer = nn.TransformerEncoder(decoder_layer, num_layers=num_layers)
        
        # 6. Multi-Output Heads
        self.head_pitch = nn.Linear(d_model, 128)
        self.head_step = nn.Linear(d_model, 100)
        self.head_dur = nn.Linear(d_model, 100)
        self.head_vel = nn.Linear(d_model, 32)

    def forward(self, p, s, d, v):
        # Input: (Batch, Seq_Len)
        
        # Embed and Fuse
        emb = torch.cat([self.pitch_emb(p), self.step_emb(s), 
                         self.dur_emb(d), self.vel_emb(v)], dim=-1)
        x = self.fusion(emb) # (B, L, D)
        
        # CNN Motif Scanner (Requires Batch, Channel, Length)
        x = x.transpose(1, 2) 
        x = self.conv_frontend(x)
        x = x.transpose(1, 2) # Back to (B, L, D)
        
        # Add Positional Information
        x = x + self.pos_emb
        
        # Transformer Processing
        x = self.transformer(x)
        
        # Predict based on the last context vector
        last_context = x[:, -1, :]
        
        return (
            self.head_pitch(last_context),
            self.head_step(last_context),
            self.head_dur(last_context),
            self.head_vel(last_context)
        )

## TOKENIZER

In [4]:
class MusicTokenizer:
    def __init__(self):
        # We define the boundaries for our bins (100 bins)
        # For time, we use log-spacing because 0.01s vs 0.02s is more important than 3.0s vs 3.01s
        self.time_bins = np.logspace(np.log10(0.001), np.log10(4.0), num=100)
        
    def tokenize(self, pitch_raw, step_raw, dur_raw, vel_raw):
        """
        Converts normalized or raw values into token indices.
        Input: Arrays of values
        """
        # 1. Pitch: Simply as an integer (0-127)
        pitch_tokens = np.clip(pitch_raw, 0, 127).astype(int)
        
        # 2. Step & Duration: Sort into bins (0-99)
        step_tokens = np.digitize(step_raw, self.time_bins) - 1
        dur_tokens = np.digitize(dur_raw, self.time_bins) - 1
        
        # 3. Velocity: Reduce 128 values down to 32 bins (0-31)
        vel_tokens = (vel_raw // 4).astype(int)
        
        # Clip to ensure we stay within the vocabulary range
        return (
            np.clip(pitch_tokens, 0, 127),
            np.clip(step_tokens, 0, 99),
            np.clip(dur_tokens, 0, 99),
            np.clip(vel_tokens, 0, 31)
        )

    def detokenize(self, p_idx, s_idx, d_idx, v_idx):
        """Converts tokens back into real values for MIDI"""
        pitch = int(p_idx)
        step = self.time_bins[s_idx]
        dur = self.time_bins[d_idx]
        vel = int(v_idx * 4)
        return pitch, step, dur, vel

In [5]:
class TokenizedMusicDataset(Dataset):
    def __init__(self, data_list, window_size=64):
        """
        Args:
            data_list: List of numpy arrays, each shape (N, 4) 
                       with [pitch, step, duration, velocity]
            window_size: Number of previous notes to look at
        """
        self.window_size = window_size
        self.tokenizer = MusicTokenizer()
        
        self.p_sequences = []
        self.s_sequences = []
        self.d_sequences = []
        self.v_sequences = []
        
        self.p_targets = []
        self.s_targets = []
        self.d_targets = []
        self.v_targets = []

        for music_array in data_list:
            # 1. Tokenize the raw musical data
            p, s, d, v = self.tokenizer.tokenize(
                music_array[:, 0], 
                music_array[:, 1], 
                music_array[:, 2], 
                music_array[:, 3]
            )
            
            # 2. Create sliding window sequences
            for i in range(len(p) - window_size):
                self.p_sequences.append(p[i:i + window_size])
                self.s_sequences.append(s[i:i + window_size])
                self.d_sequences.append(d[i:i + window_size])
                self.v_sequences.append(v[i:i + window_size])
                
                # Targets are the next tokens in the sequence
                self.p_targets.append(p[i + window_size])
                self.s_targets.append(s[i + window_size])
                self.d_targets.append(d[i + window_size])
                self.v_targets.append(v[i + window_size])

    def __len__(self):
        return len(self.p_targets)

    def __getitem__(self, idx):
        # Return sequences and targets as LongTensors (required for Embedding layers)
        return {
            'seq': (
                torch.LongTensor(self.p_sequences[idx]),
                torch.LongTensor(self.s_sequences[idx]),
                torch.LongTensor(self.d_sequences[idx]),
                torch.LongTensor(self.v_sequences[idx])
            ),
            'target': (
                torch.tensor(self.p_targets[idx], dtype=torch.long),
                torch.tensor(self.s_targets[idx], dtype=torch.long),
                torch.tensor(self.d_targets[idx], dtype=torch.long),
                torch.tensor(self.v_targets[idx], dtype=torch.long)
            )
        }

In [6]:
def train_step(model, optimizer, batch, device):
    model.train()
    optimizer.zero_grad()
    
    # Extract sequences and targets from the tokenized dataset
    p_seq, s_seq, d_seq, v_seq = [x.to(device) for x in batch['seq']]
    p_tar, s_tar, d_tar, v_tar = [x.to(device) for x in batch['target']]
    
    # Forward Pass
    out_p, out_s, out_d, out_v = model(p_seq, s_seq, d_seq, v_seq)
    
    # Loss Calculation (CrossEntropy for Classification)
    criterion = nn.CrossEntropyLoss()
    
    loss_p = criterion(out_p, p_tar)
    loss_s = criterion(out_s, s_tar)
    loss_d = criterion(out_d, d_tar)
    loss_v = criterion(out_v, v_tar)
    
    # Weighted Loss: Focus more on Rhythm (Step) and Pitch
    total_loss = loss_p + (loss_s * 2.0) + (loss_d * 1.5) + (loss_v * 0.5)
    
    total_loss.backward()
    # Gradient Clipping to prevent Transformer spikes
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    return total_loss.item()

In [7]:
def train_and_validate(model, train_loader, val_loader, optimizer, device, epochs=50):
    # Initialize TensorBoard writer
    writer = SummaryWriter(log_dir='runs/music_transformer_experiment')
    criterion = nn.CrossEntropyLoss()
    
    print(f"--- Training started on {device} ---")
    
    for epoch in range(epochs):
        # --- TRAINING PHASE ---
        model.train()
        total_train_loss = 0
        
        for batch in train_loader:
            optimizer.zero_grad()
            
            # Unpack batch (Assuming TokenizedMusicDataset from previous step)
            p_seq, s_seq, d_seq, v_seq = [x.to(device) for x in batch['seq']]
            p_tar, s_tar, d_tar, v_tar = [x.to(device) for x in batch['target']]
            
            # Forward pass
            out_p, out_s, out_d, out_v = model(p_seq, s_seq, d_seq, v_seq)
            
            # Calculate multi-task loss
            loss_p = criterion(out_p, p_tar)
            loss_s = criterion(out_s, s_tar)
            loss_d = criterion(out_d, d_tar)
            loss_v = criterion(out_v, v_tar)
            
            # Weighted loss (Rhythm and Pitch prioritized)
            batch_loss = loss_p + (loss_s * 2.0) + (loss_d * 1.5) + (loss_v * 0.5)
            
            batch_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Gradient clipping
            optimizer.step()
            
            total_train_loss += batch_loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        # --- VALIDATION PHASE ---
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                p_seq, s_seq, d_seq, v_seq = [x.to(device) for x in batch['seq']]
                p_tar, s_tar, d_tar, v_tar = [x.to(device) for x in batch['target']]
                
                out_p, out_s, out_d, out_v = model(p_seq, s_seq, d_seq, v_seq)
                
                v_loss_p = criterion(out_p, p_tar)
                v_loss_s = criterion(out_s, s_tar)
                v_loss_d = criterion(out_d, d_tar)
                v_loss_v = criterion(out_v, v_tar)
                
                val_loss = v_loss_p + (v_loss_s * 2.0) + (v_loss_d * 1.5) + (v_loss_v * 0.5)
                total_val_loss += val_loss.item()
        
        avg_val_loss = total_val_loss / len(val_loader)

        # --- LOGGING & PRINTING ---
        # Print to console
        print(f"Epoch [{epoch+1:02d}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        
        # Write to TensorBoard
        writer.add_scalar('Loss/Train', avg_train_loss, epoch)
        writer.add_scalar('Loss/Validation', avg_val_loss, epoch)
        # Log individual component losses for debugging
        writer.add_scalar('Components/Pitch_Loss', loss_p.item(), epoch)
        writer.add_scalar('Components/Step_Loss', loss_s.item(), epoch)

    writer.close()
    print("--- Training Complete ---")

In [8]:
def run_training(model, optimizer, train_loader, val_loader, epochs=50, lr=1e-4):
    # Use AdamW for SOTA Transformer stability
    criterion = nn.CrossEntropyLoss()
    writer = SummaryWriter('runs/MusicTransformer_SOTA')
    
    best_val_loss = float('inf')

    # Track losses for the plotting function
    train_history = []
    val_history = []

    for epoch in range(epochs):
        # --- TRAINING ---
        model.train()
        total_train_loss = 0
        
        for batch_idx, batch in enumerate(train_loader):
            p_seq, s_seq, d_seq, v_seq = [x.to(device) for x in batch['seq']]
            p_tar, s_tar, d_tar, v_tar = [x.to(device) for x in batch['target']]
            
            optimizer.zero_grad()
            
            # Forward pass (distributed across GPUs)
            out_p, out_s, out_d, out_v = model(p_seq, s_seq, d_seq, v_seq)
            
            # Multi-task loss calculation
            loss_p = criterion(out_p, p_tar)
            loss_s = criterion(out_s, s_tar)
            loss_d = criterion(out_d, d_tar)
            loss_v = criterion(out_v, v_tar)
            
            # Prioritize rhythm and pitch via weighting
            loss = loss_p + (loss_s * 2.0) + (loss_d * 1.5) + (loss_v * 0.5)
            
            loss.backward()
            # Clip gradients to avoid Transformer instability
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        # --- VALIDATION ---
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                p_seq, s_seq, d_seq, v_seq = [x.to(device) for x in batch['seq']]
                p_tar, s_tar, d_tar, v_tar = [x.to(device) for x in batch['target']]
                
                out_p, out_s, out_d, out_v = model(p_seq, s_seq, d_seq, v_seq)
                
                v_loss = criterion(out_p, p_tar) + (criterion(out_s, s_tar) * 2.0) + \
                         (criterion(out_d, d_tar) * 1.5) + (criterion(out_v, v_tar) * 0.5)
                total_val_loss += v_loss.item()
        
        avg_val_loss = total_val_loss / len(val_loader)

        # Save to history lists
        train_history.append(avg_train_loss)
        val_history.append(avg_val_loss)

        # --- LOGGING & CHECKPOINTING ---
        print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f}")
        
        writer.add_scalar('Loss/Train', avg_train_loss, epoch)
        writer.add_scalar('Loss/Validation', avg_val_loss, epoch)

        # Save the best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            save_model(model, optimizer, epoch, avg_val_loss, "best_music_model.pth")
        
        # Save regular checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            save_model(model, optimizer, epoch, avg_val_loss, f"checkpoint_epoch_{epoch+1}.pth")
        if (epoch+1) == epochs:
            save_model(model, optimizer, epoch, avg_val_loss, f"final_model{epoch}.pth")

    writer.close()
    plot_learning_curves(train_history, val_history)

## Plotting

In [9]:
def plot_learning_curves(train_losses, val_losses):
    """
    Plots the training and validation loss curves.
    """
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.title('Training and Validation Loss Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

def visualize_predictions(model, val_loader, tokenizer, device):
    """
    Visualizes the distribution of predicted pitches vs actual pitches 
    from a single validation batch.
    """
    model.eval()
    batch = next(iter(val_loader))
    p_seq, s_seq, d_seq, v_seq = [x.to(device) for x in batch['seq']]
    p_tar = batch['target'][0].cpu().numpy()
    
    with torch.no_grad():
        out_p, _, _, _ = model(p_seq, s_seq, d_seq, v_seq)
        # Get the most likely pitch (Argmax)
        predictions = torch.argmax(out_p, dim=-1).cpu().numpy()

    plt.figure(figsize=(12, 4))
    plt.scatter(range(len(p_tar)), p_tar, label='Actual Pitch', alpha=0.6)
    plt.scatter(range(len(predictions)), predictions, label='Predicted Pitch', marker='x')
    plt.title('Pitch Prediction Comparison (Val Batch)')
    plt.xlabel('Sample Index')
    plt.ylabel('MIDI Pitch')
    plt.legend()
    plt.show()

## Reprepare MIDI Files

In [10]:
def find_midi_files(root_path):
    # Searches recursively for all midi files
    search_pattern = os.path.join(root_path, "**/*.mid*")
    files = glob.glob(search_pattern, recursive=True)
    print(f"Found {len(files)} MIDI files in {root_path}")
    return files

In [11]:
class MaestroTokenizer:
    def __init__(self):
        # 100 log-spaced bins for time-related features (0.001s to 4s)
        self.time_bins = np.logspace(np.log10(0.001), np.log10(4.0), num=100)

    def tokenize_file(self, file_path):
        try:
            pm = pretty_midi.PrettyMIDI(file_path)
            # MAESTRO is usually all piano, but we merge just in case
            notes = []
            for instrument in pm.instruments:
                if not instrument.is_drum:
                    notes.extend(instrument.notes)
            
            # Sort notes by start time
            notes.sort(key=lambda x: x.start)
            
            tokenized_events = []
            for i in range(len(notes)):
                n = notes[i]
                
                # Feature 1: Pitch (0-127)
                p_idx = int(np.clip(n.pitch, 0, 127))
                
                # Feature 2: Step (Time since previous note started)
                step = n.start - notes[i-1].start if i > 0 else 0
                s_idx = np.clip(np.digitize(step, self.time_bins) - 1, 0, 99)
                
                # Feature 3: Duration
                dur = n.end - n.start
                d_idx = np.clip(np.digitize(dur, self.time_bins) - 1, 0, 99)
                
                # Feature 4: Velocity (Quantized 128 -> 32)
                v_idx = int(np.clip(n.velocity // 4, 0, 31))
                
                tokenized_events.append([p_idx, s_idx, d_idx, v_idx])
                
            return np.array(tokenized_events, dtype=np.int64)
        except Exception as e:
            # print(f"Error processing {file_path}: {e}")
            return None

In [12]:
def build_maestro_dataset(root_path, tokenizer, max_files=50):
    """
    Finds and tokenizes MIDI files using pretty_midi.
    max_files: Limits the number of files processed to save memory/time.
    """
    # Recursive search for all MIDI variations
    files = glob.glob(os.path.join(root_path, "**/*.mid*"), recursive=True)
    
    # Shuffle or just slice to get a representative sample
    if max_files and max_files < len(files):
        print(f"Limiting dataset to {max_files} files (out of {len(files)} found).")
        files = files[:max_files]
        
    all_data = []
    print(f"Processing {len(files)} MIDI files using pretty_midi...")
    
    for f in tqdm(files):
        tokens = tokenizer.tokenize_file(f)
        # We only keep files that actually have musical content
        if tokens is not None and len(tokens) > 65: 
            all_data.append(tokens)
            
    return all_data

In [13]:
class TokenizedMusicDataset(Dataset):
    def __init__(self, data_list, window_size=64):
        self.window_size = window_size
        self.samples = []

        for sequence in data_list:
            # sequence shape: (N_notes, 4)
            for i in range(len(sequence) - window_size):
                window = sequence[i : i + window_size]
                target = sequence[i + window_size]
                self.samples.append((window, target))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        window, target = self.samples[idx]
        
        # Split features into separate LongTensors for the Embeddings
        # Window: (64, 4) -> 4 separate (64,)
        p_seq = torch.LongTensor(window[:, 0])
        s_seq = torch.LongTensor(window[:, 1])
        d_seq = torch.LongTensor(window[:, 2])
        v_seq = torch.LongTensor(window[:, 3])
        
        # Targets are individual scalars (Indices)
        p_tar = torch.tensor(target[0], dtype=torch.long)
        s_tar = torch.tensor(target[1], dtype=torch.long)
        d_tar = torch.tensor(target[2], dtype=torch.long)
        v_tar = torch.tensor(target[3], dtype=torch.long)
        
        return {
            'seq': (p_seq, s_seq, d_seq, v_seq),
            'target': (p_tar, s_tar, d_tar, v_tar)
        }

In [14]:
def save_model(model, optimizer, epoch, loss, filename):
    """
    Saves the model state, handling DataParallel wrapping.
    """
    # Check if model is wrapped in DataParallel
    if hasattr(model, 'module'):
        state_dict = model.module.state_dict()
    else:
        state_dict = model.state_dict()
        
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': state_dict,
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }
    torch.save(checkpoint, filename)
    print(f"--> Saved checkpoint: {filename}")

In [15]:
# --- CONFIGURATION ---
MAESTRO_PATH = "/home/nour_mbb/assignment3/datasets/midi/maestro/maestro-v3.0.0" # Adjust this to your folder
WINDOW_SIZE = 64
BATCH_SIZE = 192
MAX_FILES = 150 # Set to None for full training

# 1. Initialize our components
tokenizer = MaestroTokenizer()

# 2. Build the tokenized data list
raw_token_data = build_maestro_dataset(MAESTRO_PATH, tokenizer, max_files=MAX_FILES)

# 3. Create the PyTorch Dataset
# This will generate thousands of sliding-window samples from your 50 files
full_ds = TokenizedMusicDataset(raw_token_data, window_size=WINDOW_SIZE)

# 4. Perform the Train/Val Split (90% Train, 10% Validation)
train_size = int(0.9 * len(full_ds))
val_size = len(full_ds) - train_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size])

# 5. DataLoaders for the training loop
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nSetup Complete:")
print(f"Total Sequences: {len(full_ds)}")
print(f"Training on: {len(train_ds)} samples")
print(f"Validating on: {len(val_ds)} samples")

Limiting dataset to 150 files (out of 1276 found).
Processing 150 MIDI files using pretty_midi...


100%|█████████████████████████████████████████| 150/150 [01:20<00:00,  1.87it/s]



Setup Complete:
Total Sequences: 758963
Training on: 683066 samples
Validating on: 75897 samples


In [16]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2,3"
print(f"Verfügbare GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"Gerät {i}: {torch.cuda.get_device_name(i)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Verfügbare GPUs: 3
Gerät 0: NVIDIA GeForce GTX 1080
Gerät 1: NVIDIA GeForce GTX 1080
Gerät 2: NVIDIA GeForce GTX 1080


In [17]:
EPOCHS = 40
LR = 0.0001 # Transformers prefer lower learning rates

# Initialize model
# model = SotaMusicTransformerCNN(d_model=256, nhead=8, num_layers=6).to(device)
model = MusicSotaTransformer(d_model=256, nhead=8, num_layers=6).to(device)

if torch.cuda.device_count() > 1:
    # Wrap model to use all 3 GPUs
    model = nn.DataParallel(model, device_ids=[0, 1, 2])

model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

# Run the loop
run_training(model, optimizer, train_loader, val_loader, epochs=EPOCHS)

Epoch [1/40] - Train Loss: 16.9801 - Val Loss: 16.3233
--> Saved checkpoint: best_music_model.pth
Epoch [2/40] - Train Loss: 16.0722 - Val Loss: 15.8563
--> Saved checkpoint: best_music_model.pth
Epoch [3/40] - Train Loss: 15.6954 - Val Loss: 15.5962
--> Saved checkpoint: best_music_model.pth
Epoch [4/40] - Train Loss: 15.4307 - Val Loss: 15.4074
--> Saved checkpoint: best_music_model.pth
Epoch [5/40] - Train Loss: 15.2145 - Val Loss: 15.3000
--> Saved checkpoint: best_music_model.pth
Epoch [6/40] - Train Loss: 15.0305 - Val Loss: 15.1964
--> Saved checkpoint: best_music_model.pth
Epoch [7/40] - Train Loss: 14.8628 - Val Loss: 15.1116
--> Saved checkpoint: best_music_model.pth
Epoch [8/40] - Train Loss: 14.7083 - Val Loss: 15.0621
--> Saved checkpoint: best_music_model.pth
Epoch [9/40] - Train Loss: 14.5647 - Val Loss: 15.0036
--> Saved checkpoint: best_music_model.pth
Epoch [10/40] - Train Loss: 14.4310 - Val Loss: 14.9657
--> Saved checkpoint: best_music_model.pth
--> Saved checkpoin

KeyboardInterrupt: 

In [ ]:
def generate_music(model, tokenizer, seed_seq, gen_length=200, temperature=1.0, device='cuda'):
    """
    model: Your trained SotaMusicTransformer
    tokenizer: The MaestroTokenizer used during training
    seed_seq: A starting sequence of tokens shape (64, 4)
    gen_length: How many new notes to generate
    temperature: Higher = more "creative/chaotic", Lower = more "conservative"
    """
    model.eval()
    # Initial sequence from seed
    generated_tokens = list(seed_seq) 
    
    with torch.no_grad():
        for _ in range(gen_length):
            # Take the last WINDOW_SIZE notes
            input_window = np.array(generated_tokens[-64:])
            
            # Prepare tensors
            p = torch.LongTensor(input_window[:, 0]).unsqueeze(0).to(device)
            s = torch.LongTensor(input_window[:, 1]).unsqueeze(0).to(device)
            d = torch.LongTensor(input_window[:, 2]).unsqueeze(0).to(device)
            v = torch.LongTensor(input_window[:, 3]).unsqueeze(0).to(device)
            
            # Forward pass
            out_p, out_s, out_d, out_v = model(p, s, d, v)
            
            # Apply Temperature and Sample (instead of just Argmax)
            def sample(logits, temp):
                probs = F.softmax(logits / temp, dim=-1)
                return torch.multinomial(probs, 1).item()

            # Sample each feature
            next_note = [
                sample(out_p, temperature),
                sample(out_s, temperature),
                sample(out_d, temperature),
                sample(out_v, temperature)
            ]
            
            generated_tokens.append(next_note)
            
    return np.array(generated_tokens)

In [ ]:
def tokens_to_midi(tokens, tokenizer, output_file="output.mid"):
    pm = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0) # Acoustic Grand Piano
    
    current_time = 0
    
    for tok in tokens:
        p_idx, s_idx, d_idx, v_idx = tok
        
        # --- DETOKENIZATION LOGIC ---
        
        # 1. Pitch: Direct mapping
        pitch = int(p_idx)
        
        # 2. Step & Duration: Map indices back to the time_bins values
        # We use the index to grab the corresponding value from your logspace array
        step = tokenizer.time_bins[s_idx]
        duration = tokenizer.time_bins[d_idx]

        duration = duration * 2
        step = step * 2
        
        # 3. Velocity: Reverse the // 4 quantization
        # We multiply by 4 (and add 2 to hit the middle of the quantized range)
        velocity = int(v_idx * 4 + 2)
        
        # --- MIDI CONSTRUCTION ---
        
        # Calculate start and end times
        start_time = current_time + step
        end_time = start_time + duration
        
        # Safety check: Ensure start < end
        if end_time <= start_time:
            end_time = start_time + 0.1
            
        note = pretty_midi.Note(
            velocity=np.clip(velocity, 0, 127),
            pitch=np.clip(pitch, 0, 127),
            start=float(start_time),
            end=float(end_time)
        )
        piano.notes.append(note)
        
        # Update pointer (the "step" is relative to the previous note's start)
        current_time = start_time
        
    pm.instruments.append(piano)
    pm.write(output_file)
    print(f"Successfully saved MIDI to {output_file}")

In [ ]:
# 1. Get a seed from the validation set
import random
TEMPERATURE = 0.7
rand = random.randint(0,6)
sample_data = val_ds[rand]
seed = sample_data['seq'] # This is a tuple of 4 tensors
# Combine them back into a (64, 4) numpy array for the generator
seed_np = np.stack([s.numpy() for s in seed], axis=1)

# 2. Generate
# Note: If your model is wrapped in DataParallel, use model.module
raw_model = model.module if hasattr(model, 'module') else model
gen_tokens = generate_music(raw_model, tokenizer, seed_np, gen_length=300, temperature=TEMPERATURE, device=device)

# 3. Save
tokens_to_midi(gen_tokens, tokenizer, "transformer_composition.mid")

In [ ]:
# --- SETTINGS ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_FILE = "best_music_model.pth"

# 1. Load the model (kwargs must match what you used for training)
model = load_model(
    checkpoint_path=CHECKPOINT_FILE,
    model_class=MusicSotaTransformer,
    device=DEVICE,
    d_model=256, 
    nhead=8, 
    num_layers=6, 
    max_seq_len=128
)

# 2. Get a seed (using only one sample from the dataset)
# We don't need the full val_loader here, just one batch
sample_batch = next(iter(val_loader)) 
seed_np = np.stack([s[0].numpy() for s in sample_batch['seq']], axis=1) # (64, 4)

# 3. Generate
gen_tokens = generate_music(model, tokenizer, seed_np, gen_length=400, temperature=0.8, device=DEVICE)

# 4. Save to MIDI
tokens_to_midi(gen_tokens, tokenizer, "loaded_model_output.mid")